# 03 · MicroDuck in Isaac Sim 6.2
**15 minutes.** Objective: start the real Newton-owned MicroDuck, inspect its onboard image and measured sensors, and distinguish those from evaluator-only state.

Isaac Lab 3.0 owns the Newton model and stepping; Isaac Sim 6.2 provides application/rendering services. We do **not** create a second PhysX articulation. The public pretrained actor is used **only for this introductory zero-command sensor episode**, before training. Notebook05 must instead select your exact Notebook04 export.

This is an explicit operator episode launch. No agent or motion command is issued. Policy balance still actuates joints under gravity: zero desired velocity is not torque-off. Leave unrelated GPU workloads alone.

In [ ]:
from pathlib import Path
import os, sys, json
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Select the host Python 3.12 kernel created by scripts/notebook_setup.sh")
# This kernel is deliberately not a ROS / Kit interpreter.
start = Path(os.environ.get("WORKSHOP_ROOT", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p / "notebooks/workshop.py").is_file()
             and (p / "src/microduck_workshop").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository, or set WORKSHOP_ROOT")
sys.path.insert(0, str(ROOT / "notebooks"))
from workshop import Session
session = Session(ROOT, work=os.environ.get("WORKSHOP_NOTEBOOK_WORK"))
VALIDATE = os.environ.get("WORKSHOP_NOTEBOOK_MODE") == "validate"
print("Host Python:", sys.version.split()[0])
print("Mode:", "safe validation (hardware SKIPPED)" if VALIDATE else "interactive")

## Predict · 2 minutes
Which values are legitimate policy/agent sensor inputs: joint encoders, IMU acceleration, rendered head image, perfect world root pose, human overview? Why is a simulator's true orientation not automatically an IMU measurement?

In [ ]:
RUN_SIM = False
ACCEPT_NVIDIA_EULA = False
GPU = os.environ.get("WORKSHOP_GPU", "0")
SIM_IMAGE = os.environ.get("WORKSHOP_SIM_IMAGE", "roscon/microduck-sim:local")
ROS_IMAGE = os.environ.get("WORKSHOP_ROS_IMAGE", "roscon/microduck-ros5:local")
ASSETS = Path(os.environ.get("WORKSHOP_ASSET_ROOT", ROOT / ".cache/assets")).resolve()
SIM_URL = "http://127.0.0.1:8766"  # current native transport is loopback-only

In [ ]:
def start_sensor_episode():
    session.require("02")
    session.gpu_preflight(GPU, accept_eula=ACCEPT_NVIDIA_EULA)
    session.port_available(8766)  # refuses conflict; never attaches to another person's simulator
    output = session.new_output("sensors")
    session.start("sim", SIM_IMAGE,
        ["-m", "microduck_workshop.sim.play", "--asset-root", "/assets", "--mode", "serve",
         "--cameras", "--tag-id", "0", "--steps", "45000", "--output", "/outputs/episode"],
        memory="32g", cpus=8, network="host",
        options=session.sim_options(GPU, ASSETS, output))
    health = session.wait_sim(SIM_URL)  # bounded cold start + fresh state/camera; no lease yet
    print(json.dumps(health, indent=2))
    state = session.state()
    state["sensor_episode"] = {"kind": "LIVE_SENSOR_EPISODE", "episode_id": health["episode_id"],
                               "output": str(output), "policy_sha256": health["runtime"]["policy_sha256"]}
    session.save(state)  # startup is NOT notebook completion
session.step("start real Sim 6.2 / Lab 3 / Newton sensor episode", RUN_SIM, start_sensor_episode)

## Observe · 4 minutes
The image is fetched from the actual head-camera endpoint and displayed below, with capture headers saved beside it. It is not a diagram or canned screenshot. Compare its episode to the current health response. A stale image is a blocked observation, not permission to move.

A separate **human overview** is useful for you but is forbidden as agent input. We display the onboard view only. Do not infer sensor pairing from two independently sampled screenshots.

In [ ]:
def show_onboard_camera():
    from IPython.display import Image, display
    state = session.state()
    entry = state["sensor_episode"]
    path = session.camera_snapshot(SIM_URL, expected_episode=entry["episode_id"], label="03-head")
    display(Image(filename=str(path)))
    print(json.dumps(session.get_json(SIM_URL + "/v1/camera_info"), indent=2))
    state["sensor_episode"]["head_snapshot"] = str(path)
    session.save(state)
session.step("display actual rendered onboard head camera", RUN_SIM, show_onboard_camera)

## Task · 5 minutes
Only now start the complete ROS launch: it needs a live native simulator. Find the IMU's acceleration units and inspect the image header. Compare the detector's input publisher count to Notebook02.

The original monotonic capture time, paired calibration and optical frame must survive publication. Separate `--once` samples can be different frames; do not declare stamp mismatch or exact pairing from unrelated samples. The full gateway claims a one-shot lease; this **sensor lesson is not capstone admission**.

In [ ]:
def start_and_inspect_robot_ros():
    entry = session.state()["sensor_episode"]
    snapshot = Path(entry["head_snapshot"])  # run and inspect the real head-camera cell first
    session.wait_sensors(SIM_URL, expected_episode=entry["episode_id"])  # GET-only, before gateway lease
    session.start("ros", ROS_IMAGE,
        ["ros2", "launch", "microduck_workshop_ros", "workshop.launch.py",
         "sim_url:=" + SIM_URL, "tag_size:=0.20"], network="host", memory="8g", cpus=4,
        options=[*session.gpu_options(GPU), "--ipc", "shareable"])
    session.wait_ros_node("ros", "/workshop/microduck/gateway", timeout=60)
    session.ros("ros", ["topic", "info", "/workshop/microduck/camera/image_rect", "--verbose"])
    session.ros("ros", ["topic", "echo", "/workshop/microduck/imu", "sensor_msgs/msg/Imu",
                        "--once", "--qos-reliability", "best_effort"])
    session.ros("ros", ["topic", "echo", "/workshop/microduck/camera/image_rect", "sensor_msgs/msg/Image",
                        "--once", "--field", "header", "--qos-reliability", "best_effort"])
    session.complete("03", entry, artifacts=[snapshot, snapshot.with_name("capture.json")])
session.step("publish actual robot sensors through ROS", RUN_SIM, start_and_inspect_robot_ros)

## Debug · 2 minutes
- Port occupied: stop nothing automatically. Ask the owner or use a separate prepared instance. Do not attach to a random listener.
- Runtime/version mismatch: read `runtime_identity.json` in this run's output. Product Lab 3.0 may report distribution version `28.0.0`; the installed source pin is authoritative.
- Cold start: the shared read-only readiness budget defaults to 600 seconds (`WORKSHOP_SIM_STARTUP_TIMEOUT`, bounded at 1800). Prewarm this exact session's `sim-home` before class; another worker's cache does not count. This never extends the 0.5-second capture/lease limits.
- Missing image/503: native health alone is not camera readiness. The bounded gate requires fresh same-episode state and actual JPEG/capture headers. On timeout, inspect native logs and preserve failure; do not synthesize or restamp a JPEG.
- Stopped or changed episode: old state is invalid. Explicitly clean up and start a new episode; no automatic re-arm.

## Cleanup · 2 minutes
Predict what happens to sensor freshness when the source stops. Then stop this lesson's ROS and simulator before training, keeping logs and images. On a 48 GB L40S do not co-load training and the LLM. Check only your owned GPU allocations; unrelated work is not ours to kill.

**Checkpoint:** name the physics owner, one sensor unit and the image's real episode. Next: [train an actual policy](04_train_isaac_lab_3.ipynb).

In [ ]:
def cleanup_sensor_lesson():
    session.stop("ros")
    session.stop("sim")
session.step("stop owned sensors before training", RUN_SIM, cleanup_sensor_lesson)